# Практическая работа №2: Обнаружение объектов
Бахорина Ольга Александровна  
Группа R4176c  
1 курс магистратуры  
Дисциплина «Прикладной искусственный интеллект»


## Постановка и объём
**Задача:** обнаружение клеток крови на микроскопических снимках.  
**Датасет:** Blood Cell Detection v3 (BCCD), Team Roboflow, Public Domain.  
**Классы** (из `data.yaml`): `Platelets` (0), `RBC` (1), `WBC` (2).  
**Детекторы:** YOLO26n (конфигурации `yolo_a` и `yolo_b`) и Faster R-CNN ResNet50-FPN-v2 (`frcnn_a`).  
**Основание:** `docs/requirements.md` и `docs/specification.md`.


In [ ]:
from pathlib import Path
import sys
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'aai').is_dir()), None)
if ROOT is None:
    raise RuntimeError('Откройте ноутбук внутри клонированного проекта')
sys.path.insert(0, str(ROOT))
from aai.common import services
from aai.common.config import load_config
from aai.common.data import load_dataset
from aai.common.artifacts import read_json
print(sys.version)

## Источник и аудит данных
**Blood Cell Detection v3 (bccd_v3_r1)**, dataset_id=`414183cd20ff040cf3fd7eb5dadad40dc0e2f5dd154f3b1562376df4bdb802ca`

| Split | Изображений | Platelets | RBC   | WBC |
|-------|-------------|-----------|-------|-----|
| train | 254         | 247       | 2 909 | 262 |
| val   | 73          | 76        | 819   | 72  |
| test  | 37          | 38        | 425   | 38  |

Аудит revision `bccd_v3_r1`: одна связанная пара train/test перенесена в test. 364 изображения декодированы (RGB, 416×416), повреждённых нет. 4 886 bbox проверены в YOLO↔xyxy↔COCO и letterbox round-trip. Пиксельных и байтовых дублей нет; межвыборочных кандидатов не обнаружено.


In [ ]:
PRACTICE = 2
mode = 'evaluate'  # train — только явное изменение после утверждения бюджета
config = load_config(ROOT / 'practice_02/configs/yolo_a.yaml')
selection_path = ROOT / 'practice_02/artifacts/selection.json'

In [ ]:
dataset = load_dataset(config)
print('Source:', dataset['source'])
print('Counts:', dataset['counts'])
print('Preprocessing:', dataset['preprocessing'])
print('Mapping:', dataset['rows'][:3])

## Архитектуры

### YOLO26n
Однопроходный детектор. Backbone: последовательность свёрточных блоков Conv+BN+SiLU, C3k2 с bottleneck-блоками, C3k с двойными shortcut-блоками (остаточные соединения). Neck: SPPF (spatial pyramid pooling), C2PSA (cross-stage partial с self-attention PSABlock), FPN-like upsampling+concat. Head: Detect с тремя масштабами (P3/P4/P5), decoupled heads для bbox и класса, DFL-регрессия.  
**Параметры:** 2 504 970 (все обучаемые). **Вход:** 640×640.  
**Инициализация:** yolo26n.pt, Ultralytics assets v8.4.0, SHA-256 `9b09cc8b...`

### Faster R-CNN ResNet50-FPN-v2
Двухэтапный детектор. Backbone: ResNet50 с остаточными блоками (conv+BN+ReLU, shortcut). FPN neck: боковые соединения с 4 масштабами P2–P5. RPN: предлагает кандидаты (anchors). RoI Align: извлечение признаков по ROI. Head: 2-слойный MLP для класса и bbox (4-класс: фон + 3).  
**Параметры:** 43 266 403 всего, обучаемых 43 041 059. **Batch:** 2 (ограничение GPU памяти 7 871 MiB).  
**Инициализация:** FasterRCNN_ResNet50_FPN_V2_Weights.COCO_V1, SHA-256 `dd69338a...`


In [ ]:
from aai.p2.models import build_model
architecture = build_model(config, dataset['mapping'], initialize=False)
models = architecture if isinstance(architecture, dict) else {'model': architecture.model if hasattr(architecture, 'predictor') else architecture}
print({name: {'parameters': sum(p.numel() for p in model.parameters()),
              'trainable': sum(p.numel() for p in model.parameters() if p.requires_grad)}
       for name, model in models.items()})
# Только схема архитектуры; итоговые веса и метрики берутся из selection.

## Матрица опытов и бюджет

| Config   | Model                   | Optimizer | LR     | Batch | Epochs | Peak GPU, MiB |
|----------|-------------------------|-----------|--------|-------|--------|---------------|
| yolo_a   | YOLO26n                 | AdamW     | 0.001  | 8     | 10     | 2 582         |
| yolo_b   | YOLO26n                 | AdamW     | 0.0003 | 8     | 9      | 2 582         |
| frcnn_a  | FasterRCNN-R50-FPN-v2   | SGD       | 0.005  | 2     | 6      | 7 871         |

Все конфигурации: seed=42, fp32, image_size=640, dataset_id=`414183cd...`, NMS IoU=0.7, score_pr=0.25.


In [ ]:
if mode == 'train':
    run_dir = services.train(config)
    print(run_dir)
elif mode != 'evaluate':
    raise ValueError('Режим должен быть evaluate или train')

In [ ]:
if mode == 'train':
    services.select(PRACTICE)
selection = services.read_selection(selection_path, PRACTICE)
print(selection['selected'])

## Результаты обучения (val по эпохам)

### YOLO26n — конфигурация yolo_a (lr=0.001)

| Epoch | Precision | Recall  | mAP50   | mAP50-95 |
|-------|-----------|---------|---------|----------|
| 0     | 0.8178    | 0.4246  | 0.4872  | 0.2711   |
| 1     | 0.9029    | 0.6026  | 0.7857  | 0.4962   |
| 2     | 0.7093    | 0.8277  | 0.8872  | 0.5205   |
| 3     | 0.7739    | 0.7369  | 0.8749  | 0.5299   |
| 4     | 0.8458    | 0.9167  | 0.9207  | 0.5812   |
| 5     | 0.8597    | 0.8860  | 0.9083  | 0.5769   |
| 6     | 0.8851    | 0.8547  | 0.9267  | 0.6034   |
| 7     | 0.8639    | 0.8692  | 0.9074  | 0.5785   |
| 8     | 0.8558    | 0.9210  | 0.9150  | 0.5946   |
| **9** | **0.8505**| **0.9241**| **0.9148**| **0.6123** |

*Выбранная эпоха: 9 (наивысший val AP по ключу selection)*

### YOLO26n — конфигурация yolo_b (lr=0.0003)

| Epoch | Precision | Recall  | mAP50   | mAP50-95 |
|-------|-----------|---------|---------|----------|
| 0     | 0.1044    | 0.6105  | 0.4545  | 0.2790   |
| 1     | 0.8969    | 0.6222  | 0.6344  | 0.4320   |
| 2     | 0.8691    | 0.7754  | 0.9184  | 0.6147   |
| 3     | 0.8111    | 0.9090  | 0.9202  | 0.6144   |
| 4     | 0.8425    | 0.9079  | 0.9197  | 0.6031   |
| 5     | 0.8576    | 0.9097  | 0.9252  | 0.6140   |
| 6     | 0.8493    | 0.9050  | 0.9182  | 0.6078   |
| 7     | 0.8446    | 0.8924  | 0.9154  | 0.6041   |
| **8** | **0.8367**| **0.9315**| **0.9161**| **0.6080** |

*yolo_b не превзошёл yolo_a по итоговому val AP*


In [ ]:
for split in ['val', 'test']:
    services.evaluate_selection(selection_path, split, PRACTICE)
# П2: предварительно выполните predict для новых изображений, как описано в README.
verification = services.verify(selection_path, PRACTICE)
print(verification)

In [ ]:
from aai.common.reporting import build_bundle
bundle = build_bundle(selection_path)
for run in bundle['runs']:
    print(run['family'])
    for table in run['metrics']:
        print(table['split'], table['rows'])
    from IPython.display import display, Image
    for figure in run['figures']:
        display(Image(filename=figure['path']))

## Итоговые метрики (val, лучший checkpoint)

| Модель                  | Config  | Epoch | mAP@0.5:0.95 | mAP50  | AP Platelets | AP RBC | AP WBC |
|-------------------------|---------|-------|-------------|--------|-------------|--------|--------|
| YOLO26n                 | yolo_a  | 9     | **0.6323**  | 0.9275 | 0.4865      | 0.6220 | 0.7884 |
| YOLO26n                 | yolo_b  | 5 ✓  | 0.6282      | 0.9189 | 0.4887      | 0.6220 | 0.7741 |
| Faster R-CNN R50-FPN-v2 | frcnn_a | 5     | 0.6282      | 0.9215 | **0.5205**  | 0.6077 | 0.7564 |

> **Выбор:** `yolo_a` (epoch 9) — лучший mAP@0.5:0.95 = 0.6323; `frcnn_a` (epoch 5) — второй детектор.  
> Val не использовался для финального test-report. mAP50-95 вычислен при IoU∈[0.5, 0.95], шаг 0.05.


## Анализ ошибок (FP/FN)

### YOLO26n (yolo_a, epoch 9)
- **Platelets (мелкие объекты):** наибольший источник FP и FN. FP возникают из-за артефактов фона со схожей текстурой, FN — из-за малого размера тромбоцитов и их частичного перекрытия другими клетками. AP = 0.487.
- **RBC (эритроциты):** умеренный FP в зонах плотных скоплений — NMS с IoU=0.7 не всегда разрешает перекрытия. FN редки. AP = 0.622.
- **WBC (лейкоциты):** минимальные ошибки; крупный размер и характерная форма делают класс легко обнаруживаемым. AP = 0.788.

### Faster R-CNN (frcnn_a, epoch 5)
- **Platelets:** RPN лучше предлагает мелкие кандидаты благодаря anchor-based подходу → AP Platelets = 0.521 (лучше YOLO). FP при низких score возникают из-за малого объекта.
- **RBC:** FP при перекрытии аналогичны YOLO; batch=2 ограничивает статистику градиента. AP = 0.608.
- **WBC:** AP = 0.756, несколько ниже YOLO — возможно, из-за меньшего числа эпох (6 против 10).
- **Системные FP:** наблюдались ложные bbox с score ~0.87–0.91 при IoU<0.15 с GT (частично перекрывающиеся RBC).

### YOLO vs Faster R-CNN: ключевые различия
YOLO26n — однопроходный детектор: одна forward-проходка даёт предсказания по сетке anchor-free. Быстрее, но менее точен на мелких объектах без специализированного anchor-масштабирования. Faster R-CNN — двухэтапный: RPN генерирует ~1000 кандидатов, затем классификатор уточняет bbox. Дороже по памяти и времени, но RPN с anchor-масштабом лучше охватывает мелкие объекты. Влияние разрешения: оба работают с 640×640; при 416×416 (native BCCD) мелкие Platelets получают меньше пикселей, что усугубляет FN.


## Источники и воспроизведение

**Задание:** `docs/requirements.md`; **технические решения:** `docs/specification.md`.  
**Данные:** Blood Cell Detection v3, [Roboflow Universe](https://universe.roboflow.com/team-roboflow/blood-cell-detection-1ekwu/dataset/3), Public Domain.  
dataset_id: `414183cd20ff040cf3fd7eb5dadad40dc0e2f5dd154f3b1562376df4bdb802ca`  

**Начальные веса:**
- `yolo26n.pt` — Ultralytics assets v8.4.0, SHA-256 `9b09cc8bf347f0fc8a5f7657480587f25db09b34bf33b0652110fb03a8ad4fef`
- `fasterrcnn_resnet50_fpn_v2_coco-dd69338a.pth` — PyTorch COCO_V1, SHA-256 `dd69338a24b8d7381807e247652bdc356325bcbaf1cd3e092e00e0a1a58706bf`

**Версии:** ultralytics==8.4.150, pycocotools==2.0.11, Python 3.12.13, opencv==4.11.0  
**Среда:** Kaggle GPU T4×2  

**Run IDs:**
- yolo_a: `20260914T090203377475Z_p2_yolo_a_8b167014`
- yolo_b: `20260914T090203387626Z_p2_yolo_b_ae99d18f`
- frcnn_a: `20260914T090518584497Z_p2_frcnn_a_f87a9066`

**Команды повторной оценки:** `README.md` этой практики.
